# 9.2 Map equation similarity

This notebook accompanies Section 9.2 of [*Community Detection with the Map Equation and Infomap: Theory and Applications*](https://doi.org/10.1145/3779648).

Map equation similarity considers the modular flow statistics identified by the map equation. These flow statistics tell us at what rate a random walker transitions along each link, whether that link exists or not, an approach that allows predicting links with the map equation. For a link that is used at rate $r$, we can compute how many bits are required to describe this transition: $\log_2(r)$.

In [ ]:
%matplotlib inline

from infomap import Infomap
from mapsim import MapSim
from _support import get_map_equation_example_network

import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# we use the "standard" map equation example network and plot it below
# the network is weighted and undirected
G, pos = get_map_equation_example_network()

In [ ]:
im = Infomap(silent=True, two_level=True, num_trials=10, seed=123)
im.add_networkx_graph(G)
im.run()
modules = dict(im.modules)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

nx.draw_networkx_nodes(
    G=G,
    pos=pos,
    ax=ax,
    node_color=[sns.color_palette()[modules[node] - 1] for node in sorted(G.nodes)],
)
nx.draw_networkx_edges(
    G=G,
    pos=pos,
    ax=ax,
    edgelist=sorted(G.edges),
    width=[G.get_edge_data(*e)["weight"] for e in sorted(G.edges)],
    alpha=0.75,
    arrows=True,
    connectionstyle="arc3,rad=0.1",
    edge_color="grey",
)
nx.draw_networkx_labels(G=G, pos=pos, ax=ax)

ax.axis("off")
plt.show()

In [ ]:
ms = MapSim()
ms.from_infomap(im)

Because modules are designed such that random walkers tend to stay within modules, links inside modules are used at higher rates and are cheaper to describe.

In [ ]:
# the cost for a non-link inside the green module
u = 0
v = 1
print(f"{u}->{v}: {ms.get_path_cost_directed(u, v):.2f} bits")

In [ ]:
# mapsim scores are, in genereal, not symmetric
u = 1
v = 0
print(f"{u}->{v}: {ms.get_path_cost_directed(u, v):.2f} bits")

In [ ]:
# an existing link inside the green module
u = 0
v = 2
print(f"{u}->{v}: {ms.get_path_cost_directed(u, v):.2f} bits")

In [ ]:
# a non link between the green and blue modules
u = 0
v = 22
print(f"{u}->{v}: {ms.get_path_cost_directed(u, v):.2f} bits")

We can also check and rank all interaction partner for a specific source node.

In [ ]:
u = 0
for v, rate in sorted(
    ms.predict_interaction_rates(0).items(), key=lambda p: p[1], reverse=True
):
    print(f"{u}->{v:2}: rate={rate:.4f}, cost={-np.log2(rate):.2f} bits")